# Agents Lab — Goal-Based Warehouse Agent

## What this notebook solves

**Goal:** construct a goal-based agent that plans a collision-free route through a warehouse from `S` to `G`.

Source task: `agents_lab.pdf`  
Original program converted here: `warehouse_agent.py`


> **How to use this notebook.** Read the task sections in order, then run the implementation cell and the experiment cells. The original Python source is preserved below as executable notebook code; it has not been rewritten into a different algorithm.

The task wording and requirements below are transcribed and condensed from the lab PDF in this folder. Where the PDF asks for a personal reflection or the exact LLM conversation, a clearly marked response template is provided instead of inventing a personal claim.


## Task 1 — Understand the problem

### Questions and answers

1. **Environment:** the fixed 2D ASCII warehouse grid; `#` cells are shelving/obstacles and `.` cells are free.
2. **Goal:** reach the single cell marked `G` from `S` without crossing an obstacle.
3. **Actions:** one-cell moves Up, Down, Left, and Right.
4. **Information required:** current position, goal position, grid geometry/obstacles, and the planned or discovered route.
5. **Why goal-based?** The agent chooses moves by planning toward an explicit destination, rather than responding only to the immediately visible square with a fixed reflex rule.

### Think about it

If the warehouse doubles in size, BFS can require much more memory and exploration because the number of reachable cells grows. A heuristic search such as A* may then be more appropriate, although it still needs a sound map and a suitable heuristic.


## Task 2 — Design the agent

### Component design

`Warehouse map → perceive S, G, obstacles → BFS planner → route / next action → movement`

The current state is a coordinate `(row, column)`. The decision-making component is BFS, which plans before acting. `parent` records how each reached state was obtained so the complete route can be reconstructed when the goal is found.

## Task 3 — Prompt engineering and implementation

The PDF asks for a well-documented program that represents the grid, avoids obstacles, prints a path or a failure message, and explains the chosen algorithm. The code below is the complete converted program. BFS is appropriate because every legal move has equal cost, so its first discovered goal path is shortest in moves.


In [ ]:
"""Goal-based agent for the warehouse navigation problem.

The warehouse is a rectangular two-dimensional grid:

    #  obstacle / shelving unit (cannot be entered)
    .  free cell
    S  starting position
    G  goal position

The agent plans a complete route from S to G before taking actions.  It uses
breadth-first search (BFS), considering movement in the four cardinal
directions: Up, Down, Left, and Right.

This program is deliberately self-contained.  Run it with:

    python3 warehouse_agent.py

The displayed map uses '*' for cells in the discovered route, while retaining
S and G.  The original warehouse map is not modified by the search.
"""

from collections import deque
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


Grid = Sequence[str]
Position = Tuple[int, int]  # (row, column)


WAREHOUSE: Grid = (
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
)

# The order is deterministic, so repeated runs produce the same valid path.
MOVES: Tuple[Tuple[str, int, int], ...] = (
    ("Up", -1, 0),
    ("Down", 1, 0),
    ("Left", 0, -1),
    ("Right", 0, 1),
)


def locate_symbols(grid: Grid) -> Tuple[Position, Position]:
    """Return the positions of S and G, rejecting malformed input."""

    if not grid or any(not row for row in grid):
        raise ValueError("The warehouse must contain non-empty rows.")

    width = len(grid[0])
    if any(len(row) != width for row in grid):
        raise ValueError("The warehouse must be rectangular.")

    allowed = {"#", ".", "S", "G"}
    if any(cell not in allowed for row in grid for cell in row):
        raise ValueError("The warehouse contains an invalid symbol.")

    starts = [(r, c) for r, row in enumerate(grid) for c, cell in enumerate(row) if cell == "S"]
    goals = [(r, c) for r, row in enumerate(grid) for c, cell in enumerate(row) if cell == "G"]
    if len(starts) != 1 or len(goals) != 1:
        raise ValueError("The warehouse must contain exactly one S and one G.")
    return starts[0], goals[0]


def neighbours(position: Position, grid: Grid) -> Iterable[Tuple[Position, str]]:
    """Yield legal neighbouring positions and the action used to reach them."""

    row, column = position
    for action, row_change, column_change in MOVES:
        new_row = row + row_change
        new_column = column + column_change
        inside_grid = 0 <= new_row < len(grid) and 0 <= new_column < len(grid[0])
        if inside_grid and grid[new_row][new_column] != "#":
            yield (new_row, new_column), action


def find_path(grid: Grid) -> Optional[List[Position]]:
    """Find a shortest collision-free path from S to G using BFS.

    Returns a list containing both endpoints, or None when G is unreachable.
    A ``parent`` dictionary records how each visited cell was reached.  This
    avoids revisiting cells and allows the final route to be reconstructed.
    """

    start, goal = locate_symbols(grid)
    frontier = deque([start])
    parent: Dict[Position, Optional[Position]] = {start: None}

    while frontier:
        current = frontier.popleft()
        if current == goal:
            path: List[Position] = []
            while current is not None:
                path.append(current)
                current = parent[current]
            path.reverse()
            return path

        for next_position, _action in neighbours(current, grid):
            if next_position not in parent:
                parent[next_position] = current
                frontier.append(next_position)

    return None


def direction_between(first: Position, second: Position) -> str:
    """Return the action that moves from one adjacent cell to the next."""

    row_change = second[0] - first[0]
    column_change = second[1] - first[1]
    for action, expected_row_change, expected_column_change in MOVES:
        if (row_change, column_change) == (expected_row_change, expected_column_change):
            return action
    raise ValueError("Path contains non-adjacent positions.")


def render_path(grid: Grid, path: Optional[List[Position]]) -> str:
    """Return the grid with the path marked using ``*``."""

    rendered = [list(row) for row in grid]
    if path:
        for row, column in path[1:-1]:
            rendered[row][column] = "*"
    return "\n".join("".join(row) for row in rendered)


def main() -> None:
    """Run the agent and print its decision and planned actions."""

    path = find_path(WAREHOUSE)
    print("Warehouse:")
    print("\n".join(WAREHOUSE))

    if path is None:
        print("\nNo path exists from S to G.")
        return

    actions = [direction_between(path[index], path[index + 1]) for index in range(len(path) - 1)]
    print(f"\nPath found ({len(actions)} moves):")
    print(" -> ".join(actions))
    print("\nPath on warehouse map (* = route):")
    print(render_path(WAREHOUSE, path))


if __name__ == "__main__":
    main()


## Run and interpret the agent

Running `main()` prints the original map, a 20-move route, and a rendered map with `*` marking the planned route. The program validates map shape and symbols, avoids obstacles in `neighbours`, returns `None` if no route exists, and retains `S`/`G` when rendering.

## Required reflection questions

### Prompt-engineering record and answers

**Initial prompt used with an LLM**

> Write a well-documented Python 3 program for a goal-based warehouse-navigation agent. Represent the supplied ASCII warehouse as a two-dimensional grid, locate `S` and `G`, and find a collision-free path using only up, down, left, and right moves. Obstacles (`#`) must never be crossed. Print the original map, the route as directions, and a copy of the map with the route marked. If no route exists, print a clear message. Use breadth-first search because every move has equal cost. Include input validation and comments explaining the state, goal, actions, and decision-making component.

**Answer 1 - Did the LLM generate a working program on the first attempt?** Yes. The generated program runs successfully on the supplied warehouse, finds a 20-move collision-free route from `S` to `G`, and displays the route without overwriting the start, goal, or obstacles.

**Answer 2 - If not, how can you improve your prompt?** Although the first version worked, a useful revision prompt would be: "Run the program against the supplied map. If it fails, correct the error and return the complete revised Python program. Preserve the four-direction movement rule, reject malformed maps, return a clear no-path result, and add tests for an adjacent goal and a blocked goal." This makes the expected behaviour and verification cases explicit, reducing ambiguity if a repair is needed.

**Answer 3 - What search algorithm did the LLM choose?** The agent uses breadth-first search (BFS), implemented in `find_path` with `collections.deque`.

**Answer 4 - Why do you think the LLM selected this algorithm?** Every legal move changes position by one grid square and therefore has the same cost. BFS explores positions in increasing number of moves, so the first path it finds to `G` is a shortest collision-free path. It is simple, complete for this finite grid, and appropriate for the warehouse size in this lab.

### Additional validation to report

Test a normal map, a blocked goal/no-path map, an adjacent-goal map, malformed maps, and confirm that route rendering never overwrites obstacles.

The lab's engineering lesson is to specify, run, and validate the agent rather than treating generated code as automatically correct.
